### References for graph state optimization

G. S. Hartnett, D. Kielpinski, S. Maity, P. S. Mundada, Y. Baum, and M. R. Hush, Automated discovery of heralded ballistic graph state generators for fusion-based photonic quantum computation, Physical Review A 113, 042608 (2026).

S. Bartolucci, P. M. Birchall, M. Gimeno-Segovia,
E. Johnston, K. Kieling, M. Pant, T. Rudolph, J. Smith,
C. Sparrow, and M. D. Vidrighin, Creation of Entangled
Photonic States Using Linear Optics (2021).

In [ ]:
import itertools

import numpy as np

from qoptcraft import Fock, PureState, photon_basis
from qoptcraft.optimization import (
    MultiHeraldStatePrep,
    MultiHeraldGubarevCost, MultiHeraldPiecewiseInfidelityCost, MultiHeraldBuresAngleCost,
    BFGS, Adam
)

import sys, os
sys.path.insert(0, os.path.abspath('..'))
from results_table import examples_table

In [ ]:
def dual_rail(bits):
    """Dual-rail Fock pattern of a qubit bitstring: qubit i lives on modes (2i, 2i+1),
    with its photon in the first mode if the bit is 0 and in the second if it is 1."""
    return tuple(mode for bit in bits for mode in ((0, 1) if bit else (1, 0)))


def pauli_orbit(n_qubits, edges):
    """Pauli orbit of a graph state: the 2^n states Z^s |G>, dual-rail encoded.

    |G> = 2^(-n/2) sum_x (-1)^{sum_{(i,j) in E} x_i x_j} |x>, and Z^s flips the sign of |x>
    once per qubit set in both s and x, so every orbit member reuses the same Fock states
    with different signs. All of them count as a success: the byproduct Z^s is undone by
    relabelling the later measurements.
    """
    bitstrings = list(itertools.product((0, 1), repeat=n_qubits))
    fock_states = [dual_rail(bits) for bits in bitstrings]
    graph_signs = np.array([(-1) ** sum(bits[i] * bits[j] for i, j in edges) for bits in bitstrings])
    orbit = []
    for pauli_bits in bitstrings:
        pauli_signs = np.array([(-1) ** np.dot(pauli_bits, bits) for bits in bitstrings])
        orbit.append(PureState(fock_states, graph_signs * pauli_signs))
    return orbit


def single_occupancy_heralds(in_state, target_states):
    """Herald patterns with at most one photon per mode."""
    target = target_states[0]
    herald_modes = in_state.modes - target.modes
    herald_photons = in_state.photons - target.photons
    return [tuple(1 if mode in occupied else 0 for mode in range(herald_modes))
            for occupied in itertools.combinations(range(herald_modes), herald_photons)]


def herald_basis(in_state, target_states):
    """Every herald pattern the problem allows, bunched outcomes included.

    The ancillas detect whatever `in_state` carries and the targets do not, so the herald's
    modes and photons are fixed by the two of them — photon number is conserved.
    """
    target = target_states[0]
    herald_modes = in_state.modes - target.modes
    herald_photons = in_state.photons - target.photons
    return photon_basis(herald_modes, herald_photons)

In [ ]:
optimizer = BFGS(max_iter=5000, line_search="wolfe")

# Graph state linear

Instead of searching for all the heralds, we reduce the subset to single occupancy heralds, which already give the optimal success probability.

In [ ]:
in_state = Fock(1,1,1,1,1,1) * Fock(0,0,0,0,0)
target_states = pauli_orbit(3, [(0,1), (1,2)])    # path 0-1-2
heralds = single_occupancy_heralds(in_state, target_states)
# heralds = herald_basis(in_state, target_states)
problem = MultiHeraldStatePrep(in_state, target_states, heralds)
cost_fun = MultiHeraldGubarevCost(problem, alpha=0.5, beta=5)

In [ ]:
N_RUNS = 1000
all_results = {
    "Riemannian optimizer": [
        optimizer.minimize(cost_fun, cost_fun.manifold.random_point(), verbose=False)
        for _ in range(N_RUNS)
    ]
}

In [ ]:
PROB_THRESH = 1.852e-2
examples_table(all_results, problem)

# Graph state $K_3$

In [ ]:
in_state = Fock(1,1,1,1,1,1) * Fock(0,0,0,0,0,0)  # 12 modes, 6 photons
target_states = pauli_orbit(3, [(0,1), (0,2), (1,2)])    # complete graph K3
# heralds = [dual_rail(bits) for bits in itertools.product((0,1), repeat=3)]
heralds = herald_basis(in_state, target_states)
problem = MultiHeraldStatePrep(in_state, target_states, heralds)
# cost_fun = MultiHeraldGubarevCost(problem, alpha=0.5, beta=5)
cost_fun = MultiHeraldBuresAngleCost(problem)

In [ ]:
N_RUNS = 1000
all_results = {
    "Riemannian optimizer": [
        optimizer.minimize(cost_fun, cost_fun.manifold.random_point(), verbose=False)
        for _ in range(N_RUNS)
    ]
}

In [ ]:
PROB_THRESH = 3.125e-2    # Hartnett et al.: 3.125% = 1/32 over 8 heralded states
examples_table(all_results, problem)